# boolean-mask-combine composite — cx27: mask the linalg.solve survivors — finite AND in-range

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `boolean-mask-combine`, `linalg-solve-batched`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "boolean-mask-combine"
DD_ATOM_IDS = ["boolean-mask-combine", "linalg-solve-batched"]
DD_SUBTOPICS = ["Numpy: Boolean mask combine", "PyTorch: Batched linalg.solve"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA's triangle-intersection pipeline solves a batched `(B, 3, 3)` linear system to recover `(u, v, w)` barycentric coordinates per (ray, triangle) candidate. Two failure modes:
- **Non-finite outputs** — even if the system isn't exactly singular, near-singular slices can return inf/nan. We mask those out with `t.isfinite(x).all(dim=-1)`.
- **Out-of-range outputs** — even a clean solve can return `u, v` outside `[0, 1]`, which means the intersection is outside the triangle. We mask those out with `(0 <= u) & (u <= 1) & (0 <= v) & (v <= 1)`.

The composition: do the batched solve first, then AND the two predicates into a single `(B,)` boolean — the rays that survive both checks.

**Anatomy.**
1. `x = t.linalg.solve(A, b)` — solve every slice at once, shape `(B, 3)`.
2. `finite = t.isfinite(x).all(dim=-1)` — per-slice finiteness check.
3. `in_range = (x >= 0).all(dim=-1) & (x <= 1).all(dim=-1)` — per-slice range check.
4. `valid = finite & in_range` — boolean-AND combine.

The `& ` is doing real work: it collapses two `(B,)` predicates into a single mask the caller can use to index back into the original batch.

### Composite Exercise — mask the linalg.solve survivors — finite AND in-range

**Atoms exercised together**: `boolean-mask-combine`, `linalg-solve-batched`

Implement `cx27_solve_and_mask(A, b)`.

- `A`: float tensor of shape `(B, N, N)`. May include ill-conditioned slices, but no slice is exactly singular (`det == 0`) — that's the job of cx29.
- `b`: float tensor of shape `(B, N)`.

Return `(x, valid)`:
- `x`: solve result, shape `(B, N)`. NaNs / infs in the raw output are allowed; do NOT post-process them.
- `valid`: boolean tensor of shape `(B,)` where `valid[i]` is True iff `x[i]` is entirely finite AND all of its entries are in `[0, 1]`.

1. **Batched solve** — one call to `t.linalg.solve(A, b)`. No loops.
2. **Mask combine** — AND `finite` and `in_range` into `valid`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx27_solve_and_mask(A, b):
    raise NotImplementedError

def _test_cx27():
    # Case A: clean 2x2 batch, all slices solvable AND in-range.
    A = t.tensor([
        [[1.0, 0.0], [0.0, 1.0]],
        [[2.0, 0.0], [0.0, 2.0]],
    ])
    b = t.tensor([[0.3, 0.4], [0.5, 0.5]])  # solutions: [0.3, 0.4] and [0.25, 0.25].
    x, valid = cx27_solve_and_mask(A, b)
    assert tuple(x.shape) == (2, 2)
    assert tuple(valid.shape) == (2,)
    assert valid.dtype == t.bool
    assert valid.all().item(), f'expected both valid, got {valid}'
    assert t.allclose(x, t.tensor([[0.3, 0.4], [0.25, 0.25]]), atol=1e-5)

    # Case B: out-of-range slice — solve succeeds but answer is outside [0, 1].
    A = t.tensor([
        [[1.0, 0.0], [0.0, 1.0]],
        [[1.0, 0.0], [0.0, 1.0]],
    ])
    b = t.tensor([[0.3, 0.4], [2.5, -1.0]])  # 2nd slice is way out of [0,1].
    x, valid = cx27_solve_and_mask(A, b)
    assert valid.tolist() == [True, False], f'expected [True, False], got {valid.tolist()}'

    # Case C: ill-conditioned (very large cond number) but representable — solve succeeds but
    # the result is huge in magnitude, so it should fail the in-range mask.
    A = t.tensor([
        [[1.0, 0.0], [0.0, 1.0]],
        [[1.0, 1.0], [1.0, 1.0001]],  # nearly rank-1 but not exactly singular.
        [[1.0, 0.0], [0.0, 1.0]],
    ])
    b = t.tensor([[0.2, 0.3], [1.0, 1.0], [0.5, 0.5]])
    x, valid = cx27_solve_and_mask(A, b)
    assert tuple(valid.shape) == (3,)
    # Slices 0 and 2 are clean identity solves — must be valid.
    assert valid[0].item() and valid[2].item()
    # Slice 1's solve (b = [1,1] against a near-rank-1 A whose b is in the column space) is
    # [1, 0] or [0, 1] depending on numerical luck — either way, in [0, 1], so valid.
    # What we really test here is that the solve did not crash on an ill-conditioned slice.
    # x[0] and x[2] must equal their b's exactly (identity A).
    assert t.allclose(x[0], b[0])
    assert t.allclose(x[2], b[2])

    # Case D: 3x3, larger batch — fuzz vs reference using identity A's.
    B = 8
    A = t.eye(3).expand(B, 3, 3).contiguous()
    b = t.rand(B, 3)  # solutions == b. Some will be in [0,1] (almost all here).
    x, valid = cx27_solve_and_mask(A, b)
    assert tuple(x.shape) == (B, 3)
    assert t.allclose(x, b, atol=1e-5)
    expected_valid = t.isfinite(b).all(dim=-1) & (b >= 0).all(dim=-1) & (b <= 1).all(dim=-1)
    assert t.equal(valid, expected_valid)
    _dd_passed.add('cx27')

_test_cx27()

<details><summary>Show solution — cx27</summary>

```python
def cx27_solve_and_mask(A, b):
    # Atom A (linalg-solve-batched): one call, no loops.
    x = t.linalg.solve(A, b)
    # Per-slice predicates over the last (N) axis.
    finite = t.isfinite(x).all(dim=-1)
    in_range = (x >= 0).all(dim=-1) & (x <= 1).all(dim=-1)
    # Atom B (boolean-mask-combine): AND the two (B,) predicates.
    valid = finite & in_range
    return x, valid
```

The composition is small but load-bearing: in real ARENA code `valid` is what selects the rays that actually hit a triangle. Splitting into `finite` + `in_range` makes the failure modes auditable — you can log how many slices each predicate vetoed.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx27'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx27',
        'subtopics': ["Numpy: Boolean mask combine", "PyTorch: Batched linalg.solve"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()